In [3]:
import pyarrow.parquet as pq
import os
import time

print("Script started: Re-chunking Parquet file.")
start_time = time.time()

# --- Configuration ---
# The original file with one large chunk
INPUT_FILEPATH = 'pivoted_data_all.parquet'
# The new, optimized file that will be created
OUTPUT_FILEPATH = 'pivoted_data_all_rechunked.parquet'
# The number of rows to include in each new chunk (row group)
ROWS_PER_CHUNK = 512

# --- Main Logic ---

def rechunk_parquet_file(input_path, output_path, row_group_size):
    """
    Reads a Parquet file and writes it to a new file with a specified
    row group size, making it much more efficient to read in chunks.
    """
    if not os.path.exists(input_path):
        print(f"Error: The input file was not found at {input_path}")
        return

    try:
        # Open the source file to read from
        parquet_file = pq.ParquetFile(input_path)
        
        # --- FIX: Get the Arrow schema from the file's metadata ---
        schema = parquet_file.metadata.schema.to_arrow_schema()
        
        print(f"Reading from '{input_path}' and writing to '{output_path}' with {row_group_size} rows per chunk.")

        # Open a new file to write to.
        with pq.ParquetWriter(output_path, schema) as writer:
            # Iterate through the original file's chunks (even if there's only one)
            for i in range(parquet_file.num_row_groups):
                print(f"  Processing original chunk {i+1}/{parquet_file.num_row_groups}...")
                # Read a chunk of data into a PyArrow Table (memory-efficient)
                table = parquet_file.read_row_group(i)
                # Write the table to the new file, specifying the row_group_size here.
                # The writer will automatically handle breaking the large table into smaller chunks.
                writer.write_table(table, row_group_size=row_group_size)

        print("\nRe-chunking complete.")
        
        # Verify the new file
        new_parquet_file = pq.ParquetFile(output_path)
        print(f"Successfully created new file with {new_parquet_file.num_row_groups} chunks (row groups).")


    except Exception as e:
        print(f"An error occurred during the process: {e}")
        
    end_time = time.time()
    print(f"Total script execution time: {(end_time - start_time):.2f} seconds.")


if __name__ == '__main__':
    rechunk_parquet_file(INPUT_FILEPATH, OUTPUT_FILEPATH, ROWS_PER_CHUNK)



Script started: Re-chunking Parquet file.
Reading from 'pivoted_data_all.parquet' and writing to 'pivoted_data_all_rechunked.parquet' with 512 rows per chunk.
  Processing original chunk 1/1...

Re-chunking complete.
Successfully created new file with 77 chunks (row groups).
Total script execution time: 490.82 seconds.
